# Strategy: Regime-Aware Mean Reversion

In [ ]:
# ============================================================
# S04 - Regime Aware Mean Reversion
# ============================================================

# ============================================================
# Imports
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    # --------------------------------------------------------
    # Extract Fields
    # --------------------------------------------------------

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    # --------------------------------------------------------
    # BTC Market Regime Filter
    # --------------------------------------------------------

    btc = close.sel(asset="BTC")

    btc_sma200 = qnta.sma(
        btc,
        200
    )

    bull_market = xr.where(
        btc > btc_sma200,
        1,
        0
    )

    # --------------------------------------------------------
    # Daily Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    # --------------------------------------------------------
    # Volatility Estimate
    # --------------------------------------------------------

    vol20 = qnta.std(
        ret,
        20
    )

    # --------------------------------------------------------
    # Mean Reversion Signal
    # --------------------------------------------------------

    ret5 = close / close.shift(time=5) - 1

    # Recent losers preferred
    score = (-ret5).rank("asset")

    score = score * is_liquid

    # --------------------------------------------------------
    # Select Strongest Assets
    # --------------------------------------------------------

    ranks = score.rank("asset")

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Volatility Weighting
    # --------------------------------------------------------

    inv_vol = 1 / (vol20 + 1e-6)

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize Portfolio
    # --------------------------------------------------------

    denom = weights.sum("asset")

    weights = xr.where(
        denom > 0,
        weights / denom,
        0
    )

    # --------------------------------------------------------
    # Apply BTC Regime Filter
    # --------------------------------------------------------

    weights = weights * bull_market

    return weights.fillna(0)

# ============================================================
# Generate Portfolio Weights
# ============================================================

weights = strategy(data)

# ============================================================
# Clean Weights
# ============================================================

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

# ============================================================
# Write Submission Output
# ============================================================

qnout.write(weights)

100% (15259192 of 15259192) |############| Elapsed Time: 0:00:00 Time:  0:00:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field         equity  relative_return  volatility  underwater  max_drawdown  \
time                                                                          
2026-06-05  4.422088              0.0    0.597925   -0.875433     -0.908719   
2026-06-06  4.422088              0.0    0.597847   -0.875433     -0.908719   
2026-06-07  4.422088              0.0    0.597768   -0.875433     -0.908719   
2026-06-08  4.422088              0.0    0.597690   -0.875433     -0.908719   
2026-06-09  4.422088              0.0    0.597612   -0.875433     -0.908719   

field       sharpe_ratio  mean_return  bias  instruments  avg_turnover  \
time                                                                     
2026-06-05      0.541163     0.323575   0.0         34.0      0.209214   
2026-06-06      0.541092     0.323490   0.0         34.0      0.209159   
2026-06-07      0.541021     0.323405   0.0         34.0      0.209104   
2026-06-08      0.540950     0.323320   0.0         34.0      0.209049   
2026-06-09      0.540879     0.323236   0.0         34.0      0.208994   

field       avg_holding_time  
time                          
2026-06-05           6.26974  
2026-06-06           6.26974  
2026-06-07           6.26974  
2026-06-08           6.26974  
2026-06-09           6.26974  

Final Metrics:
field
equity              4.422088
sharpe_ratio        0.540879
max_drawdown       -0.908719
avg_turnover        0.208994
avg_holding_time    6.269740
Name: 2026-06-09 00:00:00, dtype: float64